<a href="https://colab.research.google.com/github/Hassanmufezshaikh/AI-Agents/blob/main/McpAsAgentTool.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install --upgrade google-adk google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 603.0 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 1.5 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of google-auth[pyopenssl] to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 793.7/793.7 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 246.1/246.1 kB 4.3 MB/s eta 0:00:00
  Attempting uninstall: google-auth
    Found existing installation: google-auth 2.47.0
    Uninstalling google-auth-2.47.0:
      Successfully uninstalled google-auth-2.47.0
  Attempting uninstall: google-genai
    Found existing installation: google-genai 1.68.0
    Uninstalling google-genai-1.68.0:
      Successfully uninstalled google-genai-1.68.0
  Attempting uninstall: google-adk
    Found existing installation: google-adk 1.29.0
  

In [2]:
import google.adk
print(google.adk.__version__)

2.1.0


In [3]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64
print(" Tunnel Components imported successfully")



 Tunnel Components imported successfully


In [4]:
import os
from google.colab import userdata
userdata.get('gemeni')

try:
  GOOGLE_API_KEY = userdata.get('gemeni')
  os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY
  print("Gemini API Key Setup Complete")
except Exception as e :
  print("Authencation Error: Please add 'GEMENI_API_KEY' to your kaggale secrets, Details : {e}")

Gemini API Key Setup Complete


In [5]:
import uuid
from google.genai import types
from google.adk.agents import LlmAgent
from google.adk.models.google_llm import Gemini
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.tools.mcp_tool.mcp_toolset import McpToolset
from google.adk.tools.tool_context import ToolContext
from google.adk.tools.mcp_tool.mcp_session_manager import StdioConnectionParams, StreamableHTTPServerParams
from mcp import StdioServerParameters
from google.adk.apps.app import App, ResumabilityConfig
from google.adk.tools.function_tool import FunctionTool
print(" ADK components imported successfully.")


 ADK components imported successfully.


/usr/local/lib/python3.12/dist-packages/google/adk/features/_feature_decorator.py:72: UserWarning: [EXPERIMENTAL] feature FeatureName.PLUGGABLE_AUTH is enabled.
  check_feature_enabled()


In [6]:
from google.adk.agents import (
    Agent,
    # SequentialAgent, # Deprecated
    ParallelAgent,
    LoopAgent
)
# from google.adk import Workflow # Attempt to import Workflow directly from google.adk

from google.adk.models.google_llm import Gemini
from google.adk.runners import InMemoryRunner
from google.adk.tools import google_search, AgentTool, FunctionTool
from google.genai import types
from google.adk.agents import LlmAgent
from google.adk.models.google_llm import Gemini
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.tools.mcp_tool.mcp_toolset import McpToolset
from google.adk.tools.tool_context import ToolContext
from google.adk.tools.mcp_tool.mcp_session_manager import StdioConnectionParams, StreamableHTTPServerParams
from mcp import StdioServerParameters
from google.adk.apps.app import App, ResumabilityConfig
from google.adk.tools.function_tool import FunctionTool
print(" ADK components imported successfully.")




 ADK components imported successfully.


In [7]:
import google.adk.tools as tools

print(dir(tools))

['APIHubToolset', 'AgentTool', 'AgentTool', 'Any', 'ApiRegistry', 'AuthToolArguments', 'BaseTool', 'DiscoveryEngineSearchTool', 'ExampleTool', 'FunctionTool', 'FunctionTool', 'LongRunningFunctionTool', 'MCPToolset', 'McpToolset', 'SearchResultMode', 'TYPE_CHECKING', 'ToolContext', 'TransferToAgentTool', 'VertexAiSearchTool', '_LAZY_MAPPING', '__all__', '__builtins__', '__cached__', '__dir__', '__doc__', '__file__', '__getattr__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '_automatic_function_calling_util', '_forwarding_artifact_service', '_function_parameter_parse_util', '_function_tool_declarations', '_gemini_schema_util', 'agent_tool', 'base_authenticated_tool', 'base_tool', 'base_toolset', 'computer_use', 'enterprise_web_search', 'exit_loop', 'function_tool', 'get_user_choice', 'google_maps_grounding', 'google_search', 'google_search', 'google_search_tool', 'importlib', 'load_artifacts', 'load_mcp_resource_tool', 'load_memory', 'logging', 'mcp_tool', 'openapi_

In [8]:
from google.genai import types

retry_config=types.HttpRetryOptions(
attempts=5, # Maximum retry attempts
exp_base=7, # Delay multiplier
initial_delay=1, # Initial delay before first retry (in seconds)
http_status_codes=[429, 500, 503, 504]
)

In [9]:
from google.colab import userdata

#MCP integration with Github Server
try:
  GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
  mcp_github_server = McpToolset(
  connection_params=StreamableHTTPServerParams(
  url="https://api.githubcopilot.com/mcp/",
  headers={
  "Authorization": f"Bearer {GITHUB_TOKEN}",
  "X-MCP-Toolsets": "all",
  "X-MCP-Readonly": "false"
  },
  ),
  )
  print(" MCP Tool created")
except Exception as e:
  print(f"Authentication Error: Please add 'GITHUB_TOKEN' to your Colab secrets, Details : {e}")


 MCP Tool created


In [10]:

github_agent = LlmAgent(
model=Gemini(model="gemini-2.5-flash", retry_options=retry_config),
name="github_agent",
instruction="""
You are a GitHub MCP agent.
Never use GitHub search DSL.
Always call typed GitHub MCP endpoints.
Always limit pagination.
Exclude pull requests unless explicitly asked.
""",
tools=[mcp_github_server],
)

In [11]:
runner = InMemoryRunner(agent=github_agent)
print("github_agent created.")


github_agent created.


In [12]:
response =  await runner.run_debug(
    "List all the Mcp Tools you can see and describe what each does",
    verbose=True,
)

/usr/local/lib/python3.12/dist-packages/google/adk/tools/mcp_tool/mcp_toolset.py:315: UserWarning: [EXPERIMENTAL] feature FeatureName._MCP_GRACEFUL_ERROR_HANDLING is enabled.
  session = await self._mcp_session_manager.create_session(


github_agent > As a GitHub MCP agent, I have access to a variety of tools to interact with GitHub resources. Here's a list of the tools I can use and what each one does:

1.  **get_branch**: Retrieves detailed information about a specific branch within a repository.
2.  **get_commit**: Fetches detailed information about a specific commit by its SHA.
3.  **get_issue**: Retrieves detailed information about a specific issue in a repository.
4.  **get_job**: Fetches details about a specific job within a workflow run.
5.  **get_organization**: Retrieves detailed information about a specific GitHub organization.
6.  **get_organization_hook**: Gets details about a specific webhook configured for an organization.
7.  **get_organization_members**: Lists all members of a specified organization.
8.  **get_organization_repositories**: Lists all public and private repositories belonging to an organization.
9.  **get_organization_secret**: Retrieves metadata for a specific secret configured at the o